# Pipeline preprocessing для CatBoost

Этот ноутбук использует ту же очистку данных и feature engineering, что и основной `preprocessing.ipynb`, но меняет финальную подготовку признаков под CatBoost.

Что отличается:
- не используем `StandardScaler`;
- не используем `OneHotEncoder`;
- числовые признаки остаются числовыми;
- числовые `NaN` оставляем намеренно, потому что CatBoost умеет обрабатывать пропуски в числовых признаках;
- категориальные признаки остаются строками, а пропуски в них заполняются отдельной категорией;
- формируются `cat_feature_names` и `cat_feature_indices` для `CatBoostClassifier` или `Pool`;
- train/test split делается по `Customer_ID`, чтобы один клиент не попадал одновременно в train и test.

Если задача будет формулироваться как прогноз будущих месяцев, вместо group split нужно использовать time-based split по `Month`: обучаться на ранних месяцах и проверяться на более поздних.


In [12]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42
TARGET = "Credit_Score"
SCORE_ORDER = ["Poor", "Standard", "Good"]
SCORE_MAP = {"Poor": 0, "Standard": 1, "Good": 2}
GROUP_COLUMN = "Customer_ID"
TEST_SIZE = 0.2
SPLIT_STRATEGY = "customer_id_group_shuffle"
NUMERIC_NAN_POLICY = "Оставляем числовые NaN для встроенной обработки CatBoost"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "train.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "train.csv"


In [13]:
train_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(train_raw.shape)
train_raw.head()

(100000, 28)


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


In [14]:
MISSING_MARKERS = {
    "",
    "_",
    "__",
    "___",
    "_______",
    "!@9#%8",
    "nan",
    "NaN",
    "None",
    "<NA>",
}

NUMERIC_LIKE_COLS = [
    "Age",
    "Annual_Income",
    "Num_of_Loan",
    "Num_of_Delayed_Payment",
    "Changed_Credit_Limit",
    "Outstanding_Debt",
    "Amount_invested_monthly",
    "Monthly_Balance",
]

EXPECTED_RANGES = {
    "Age": (18, 100),
    "Annual_Income": (0, 1_000_000),
    "Monthly_Inhand_Salary": (0, 100_000),
    "Num_Bank_Accounts": (0, 20),
    "Num_Credit_Card": (0, 20),
    "Interest_Rate": (0, 100),
    "Num_of_Loan": (0, 20),
    "Delay_from_due_date": (0, 90),
    "Num_of_Delayed_Payment": (0, 100),
    "Num_Credit_Inquiries": (0, 100),
    "Outstanding_Debt": (0, 20_000),
    "Total_EMI_per_month": (0, 20_000),
    "Amount_invested_monthly": (0, 20_000),
    "Monthly_Balance": (0, 10_000),
    "Credit_History_Age_Months": (0, 600),
}

DROP_AFTER_CLEANING = [
    "ID",
    "Customer_ID",
    "Name",
    "SSN",
    "Credit_History_Age",
    "Type_of_Loan",
    TARGET,
]

PROFILE_NUMERIC_COLS = [
    "Age",
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "Num_Bank_Accounts",
    "Num_Credit_Card",
    "Interest_Rate",
    "Num_of_Loan",
    "Num_of_Delayed_Payment",
    "Changed_Credit_Limit",
    "Num_Credit_Inquiries",
    "Outstanding_Debt",
    "Credit_History_Age_Months",
    "Total_EMI_per_month",
    "Amount_invested_monthly",
    "Monthly_Balance",
]

PROFILE_CATEGORICAL_COLS = [
    "Occupation",
    "Credit_Mix",
    "Payment_of_Min_Amount",
    "Payment_Behaviour",
]

MONTH_ORDER = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12,
}

ENGINEERED_FEATURES = [
    "debt_to_annual_income",
    "debt_to_monthly_salary",
    "emi_to_monthly_salary",
    "investment_to_monthly_salary",
    "balance_to_monthly_salary",
    "available_income_after_emi",
    "available_income_after_emi_and_investment",
    "delayed_payment_ratio",
    "avg_delay_per_delayed_payment",
    "inquiries_per_credit_account",
    "credit_cards_per_bank_account",
    "loans_per_bank_account",
    "credit_history_years",
    "credit_age_per_loan",
    "total_credit_products",
    "loan_diversity_ratio",
    "has_negative_payment_history",
    "high_utilization_flag",
    "low_balance_flag",
    "debt_per_credit_product",
    "interest_debt_pressure",
    "credit_inquiries_per_history_year",
    "credit_pressure_utilized_debt",
    "free_cash_flow_proxy",
    "debt_service_and_investment_to_salary",
    "secured_loan_count",
    "unsecured_loan_count",
    "has_no_reported_loan_type",
    "credit_mix_ordinal",
    "payment_of_min_amount_ordinal",
    "payment_min_amount_yes_flag",
    "is_missing_Type_of_Loan",
    "is_missing_Credit_History_Age",
    "is_missing_Monthly_Inhand_Salary",
    "is_missing_Credit_Mix",
    "missing_marker_count",
    "is_anomaly_Age",
    "is_anomaly_Delay_from_due_date",
    "anomaly_count",
]

ENGINEERED_CATEGORICAL_FEATURES = [
    "Payment_Spend_Level",
    "Payment_Value_Size",
]

SECURED_LOAN_TYPES = {
    "Auto Loan",
    "Home Equity Loan",
    "Mortgage Loan",
}

UNSECURED_LOAN_TYPES = {
    "Credit-Builder Loan",
    "Debt Consolidation Loan",
    "Payday Loan",
    "Personal Loan",
    "Student Loan",
}

CREDIT_MIX_ORDINAL = {
    "Bad": 0.0,
    "Standard": 1.0,
    "Good": 2.0,
}

PAYMENT_MIN_AMOUNT_ORDINAL = {
    "Yes": 0.0,
    "NM": 1.0,
    "No": 2.0,
}

MISSING_FLAG_SOURCE_COLS = [
    "Type_of_Loan",
    "Credit_History_Age",
    "Monthly_Inhand_Salary",
    "Credit_Mix",
]

ANOMALY_FLAG_COLS = [
    "Age",
    "Delay_from_due_date",
]


def normalize_missing_markers(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.mask(cleaned.isin(MISSING_MARKERS))
    return cleaned


def parse_numeric(series: pd.Series) -> pd.Series:
    cleaned = normalize_missing_markers(series)
    cleaned = cleaned.str.replace("_", "", regex=False)
    return pd.to_numeric(cleaned, errors="coerce").astype("float64")


def parse_credit_history_to_months(series: pd.Series) -> pd.Series:
    cleaned = normalize_missing_markers(series)
    parts = cleaned.str.extract(
        r"(?:(\d+)\s+Years?)?\s*(?:and\s*)?(?:(\d+)\s+Months?)?",
        expand=True,
    )
    years = pd.to_numeric(parts[0], errors="coerce")
    months = pd.to_numeric(parts[1], errors="coerce")
    parsed = years.fillna(0).mul(12).add(months.fillna(0))
    return parsed.where(parts.notna().any(axis=1))


def parse_loan_types(value: object) -> list[str]:
    if pd.isna(value):
        return []

    text = str(value).strip()
    if text in MISSING_MARKERS:
        return []

    text = re.sub(r"\band\b", ",", text, flags=re.IGNORECASE)
    loan_types = []
    for part in text.split(","):
        cleaned = part.strip().strip(".")
        if cleaned and cleaned not in MISSING_MARKERS:
            loan_types.append(cleaned)
    return loan_types


def _series_mode(series: pd.Series) -> object:
    values = series.dropna()
    if values.empty:
        return np.nan
    return values.mode(dropna=True).iloc[0]

In [15]:
class CreditScoreCleaner(BaseEstimator, TransformerMixin):
    # Очищает raw-строки Credit Score перед общим sklearn preprocessing.

    def __init__(
        self,
        use_customer_profiles: bool = True,
        clip_quantiles: tuple[float, float] = (0.01, 0.99),
    ):
        self.use_customer_profiles = use_customer_profiles
        self.clip_quantiles = clip_quantiles

    def fit(self, X: pd.DataFrame, y: object = None):
        data = self._basic_clean(X)
        self.feature_columns_in_ = list(X.columns)

        loan_counts = {}
        if "Type_of_Loan" in data.columns:
            for loans in data["Type_of_Loan"].apply(parse_loan_types):
                for loan in loans:
                    loan_counts[loan] = loan_counts.get(loan, 0) + 1
        self.loan_types_ = sorted(loan_counts)

        self.numeric_columns_ = [
            column
            for column in data.select_dtypes(include=[np.number]).columns
            if column not in [TARGET]
        ]

        self.clip_bounds_ = {}
        if self.clip_quantiles is not None:
            lower_q, upper_q = self.clip_quantiles
            for column in self.numeric_columns_:
                series = pd.to_numeric(data[column], errors="coerce")
                if series.notna().any():
                    lower, upper = series.quantile([lower_q, upper_q])
                    self.clip_bounds_[column] = (float(lower), float(upper))

        if self.use_customer_profiles and "Customer_ID" in data.columns:
            self.customer_numeric_profiles_ = self._fit_customer_numeric_profiles(data)
            self.customer_categorical_profiles_ = self._fit_customer_categorical_profiles(data)
        else:
            self.customer_numeric_profiles_ = pd.DataFrame()
            self.customer_categorical_profiles_ = pd.DataFrame()

        cleaned = self._apply_customer_profiles(data)
        cleaned = self._add_loan_features(cleaned)
        cleaned = self._clip_numeric_features(cleaned)
        cleaned = self._add_credit_engineering_features(cleaned)
        cleaned = self._post_clean(cleaned)
        self.output_columns_ = list(cleaned.columns)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        data = self._basic_clean(X)
        data = self._apply_customer_profiles(data)
        data = self._add_loan_features(data)
        data = self._clip_numeric_features(data)
        data = self._add_credit_engineering_features(data)
        data = self._post_clean(data)

        for column in self.output_columns_:
            if column not in data.columns:
                data[column] = np.nan
        return data[self.output_columns_]

    def _basic_clean(self, X: pd.DataFrame) -> pd.DataFrame:
        data = X.copy()

        for column in data.select_dtypes(include=["object", "string"]).columns:
            data[column] = normalize_missing_markers(data[column])

        missing_flag_columns = [column for column in MISSING_FLAG_SOURCE_COLS if column in data.columns]
        if missing_flag_columns:
            data["missing_marker_count"] = data[missing_flag_columns].isna().sum(axis=1).astype("float64")
        else:
            data["missing_marker_count"] = np.nan

        for column in MISSING_FLAG_SOURCE_COLS:
            flag_name = f"is_missing_{column}"
            if column in data.columns:
                data[flag_name] = data[column].isna().astype("float64")
            else:
                data[flag_name] = np.nan

        for column in NUMERIC_LIKE_COLS:
            if column in data.columns:
                data[column] = parse_numeric(data[column])

        if "Credit_History_Age" in data.columns:
            data["Credit_History_Age_Months"] = parse_credit_history_to_months(
                data["Credit_History_Age"]
            )

        if "Month" in data.columns:
            data["Month_Num"] = data["Month"].map(MONTH_ORDER).astype("float64")

        anomaly_count = pd.Series(0.0, index=data.index, dtype="float64")
        for column, (lower, upper) in EXPECTED_RANGES.items():
            if column in data.columns:
                series = pd.to_numeric(data[column], errors="coerce")
                anomaly_mask = series.notna() & ((series < lower) | (series > upper))
                anomaly_count = anomaly_count.add(anomaly_mask.astype("float64"), fill_value=0)
                if column in ANOMALY_FLAG_COLS:
                    data[f"is_anomaly_{column}"] = anomaly_mask.astype("float64")
                data[column] = series.mask(anomaly_mask).astype("float64")

        for column in ANOMALY_FLAG_COLS:
            flag_name = f"is_anomaly_{column}"
            if flag_name not in data.columns:
                data[flag_name] = np.nan

        data["anomaly_count"] = anomaly_count.astype("float64")
        return data

    def _fit_customer_numeric_profiles(self, data: pd.DataFrame) -> pd.DataFrame:
        columns = [
            column
            for column in PROFILE_NUMERIC_COLS
            if column in data.columns and data[column].notna().any()
        ]
        if not columns:
            return pd.DataFrame()
        return data.groupby("Customer_ID", dropna=True)[columns].median()

    def _fit_customer_categorical_profiles(self, data: pd.DataFrame) -> pd.DataFrame:
        columns = [
            column
            for column in PROFILE_CATEGORICAL_COLS
            if column in data.columns and data[column].notna().any()
        ]
        if not columns:
            return pd.DataFrame()
        return data.groupby("Customer_ID", dropna=True)[columns].agg(_series_mode)

    def _apply_customer_profiles(self, data: pd.DataFrame) -> pd.DataFrame:
        if "Customer_ID" not in data.columns:
            return data

        if not self.customer_numeric_profiles_.empty:
            for column in self.customer_numeric_profiles_.columns:
                if column in data.columns:
                    profile_values = data["Customer_ID"].map(self.customer_numeric_profiles_[column])
                    data[column] = data[column].fillna(profile_values)

        if not self.customer_categorical_profiles_.empty:
            for column in self.customer_categorical_profiles_.columns:
                if column in data.columns:
                    profile_values = data["Customer_ID"].map(self.customer_categorical_profiles_[column])
                    data[column] = data[column].fillna(profile_values)

        return data

    def _add_loan_features(self, data: pd.DataFrame) -> pd.DataFrame:
        if "Type_of_Loan" not in data.columns:
            data["Loan_Type_Count"] = np.nan
            data["secured_loan_count"] = np.nan
            data["unsecured_loan_count"] = np.nan
            data["has_no_reported_loan_type"] = np.nan
            for loan_type in self.loan_types_:
                data[f"Loan_Type__{loan_type}"] = 0.0
            return data

        parsed = data["Type_of_Loan"].apply(parse_loan_types)
        loan_sets = parsed.map(set)

        data["Loan_Type_Count"] = parsed.map(len).astype("float64")
        data["secured_loan_count"] = parsed.map(
            lambda loans: sum(loan in SECURED_LOAN_TYPES for loan in loans)
        ).astype("float64")
        data["unsecured_loan_count"] = parsed.map(
            lambda loans: sum(loan in UNSECURED_LOAN_TYPES for loan in loans)
        ).astype("float64")
        data["has_no_reported_loan_type"] = parsed.map(lambda loans: len(loans) == 0).astype("float64")

        for loan_type in self.loan_types_:
            data[f"Loan_Type__{loan_type}"] = loan_sets.map(lambda loans: float(loan_type in loans))
        return data

    def _numeric_feature(self, data: pd.DataFrame, column: str) -> pd.Series:
        if column not in data.columns:
            return pd.Series(np.nan, index=data.index, dtype="float64")
        return pd.to_numeric(data[column], errors="coerce").astype("float64")

    @staticmethod
    def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
        denominator = denominator.where(denominator.ne(0))
        result = numerator.divide(denominator)
        return result.replace([np.inf, -np.inf], np.nan).astype("float64")

    @staticmethod
    def _sum_known(*series_list: pd.Series) -> pd.Series:
        values = pd.concat(series_list, axis=1)
        result = values.fillna(0).sum(axis=1)
        return result.where(values.notna().any(axis=1)).astype("float64")

    def _add_credit_engineering_features(self, data: pd.DataFrame) -> pd.DataFrame:
        annual_income = self._numeric_feature(data, "Annual_Income")
        monthly_salary = self._numeric_feature(data, "Monthly_Inhand_Salary")
        bank_accounts = self._numeric_feature(data, "Num_Bank_Accounts")
        credit_cards = self._numeric_feature(data, "Num_Credit_Card")
        loans = self._numeric_feature(data, "Num_of_Loan")
        delayed_payments = self._numeric_feature(data, "Num_of_Delayed_Payment")
        credit_inquiries = self._numeric_feature(data, "Num_Credit_Inquiries")
        outstanding_debt = self._numeric_feature(data, "Outstanding_Debt")
        credit_history_months = self._numeric_feature(data, "Credit_History_Age_Months")
        total_emi = self._numeric_feature(data, "Total_EMI_per_month")
        monthly_investment = self._numeric_feature(data, "Amount_invested_monthly")
        monthly_balance = self._numeric_feature(data, "Monthly_Balance")
        delay_from_due_date = self._numeric_feature(data, "Delay_from_due_date")
        utilization = self._numeric_feature(data, "Credit_Utilization_Ratio")
        interest_rate = self._numeric_feature(data, "Interest_Rate")
        loan_type_count = self._numeric_feature(data, "Loan_Type_Count")

        total_credit_products = self._sum_known(credit_cards, loans)
        credit_history_years = credit_history_months.div(12).astype("float64")
        free_cash_flow = monthly_salary.sub(total_emi).sub(monthly_investment).astype("float64")

        data["debt_to_annual_income"] = self._safe_divide(outstanding_debt, annual_income)
        data["debt_to_monthly_salary"] = self._safe_divide(outstanding_debt, monthly_salary)
        data["emi_to_monthly_salary"] = self._safe_divide(total_emi, monthly_salary)
        data["investment_to_monthly_salary"] = self._safe_divide(monthly_investment, monthly_salary)
        data["balance_to_monthly_salary"] = self._safe_divide(monthly_balance, monthly_salary)
        data["available_income_after_emi"] = monthly_salary.sub(total_emi).astype("float64")
        data["available_income_after_emi_and_investment"] = free_cash_flow
        data["free_cash_flow_proxy"] = free_cash_flow
        data["debt_service_and_investment_to_salary"] = self._safe_divide(
            total_emi.add(monthly_investment),
            monthly_salary,
        )
        data["interest_debt_pressure"] = outstanding_debt.mul(interest_rate).div(100).astype("float64")
        data["credit_pressure_utilized_debt"] = outstanding_debt.mul(utilization).div(100).astype("float64")
        data["credit_inquiries_per_history_year"] = self._safe_divide(
            credit_inquiries,
            credit_history_years.add(1),
        )
        data["delayed_payment_ratio"] = self._safe_divide(
            delayed_payments,
            credit_history_months,
        )
        data["avg_delay_per_delayed_payment"] = self._safe_divide(
            delay_from_due_date,
            delayed_payments,
        )
        data["inquiries_per_credit_account"] = self._safe_divide(
            credit_inquiries,
            total_credit_products,
        )
        data["credit_cards_per_bank_account"] = self._safe_divide(
            credit_cards,
            bank_accounts,
        )
        data["loans_per_bank_account"] = self._safe_divide(loans, bank_accounts)
        data["credit_history_years"] = credit_history_years
        data["credit_age_per_loan"] = self._safe_divide(credit_history_months, loans)
        data["total_credit_products"] = total_credit_products
        data["loan_diversity_ratio"] = self._safe_divide(loan_type_count, loans)

        if "Credit_Mix" in data.columns:
            data["credit_mix_ordinal"] = data["Credit_Mix"].map(CREDIT_MIX_ORDINAL).astype("float64")
        else:
            data["credit_mix_ordinal"] = np.nan

        if "Payment_of_Min_Amount" in data.columns:
            payment_min = data["Payment_of_Min_Amount"]
            data["payment_of_min_amount_ordinal"] = payment_min.map(PAYMENT_MIN_AMOUNT_ORDINAL).astype("float64")
            data["payment_min_amount_yes_flag"] = payment_min.eq("Yes").astype("float64").where(payment_min.notna())
        else:
            data["payment_of_min_amount_ordinal"] = np.nan
            data["payment_min_amount_yes_flag"] = np.nan

        if "Payment_Behaviour" in data.columns:
            behaviour = data["Payment_Behaviour"].astype("string")
            data["Payment_Spend_Level"] = behaviour.str.extract(
                r"^(High|Low)_spent",
                expand=False,
            ).astype("object")
            data["Payment_Value_Size"] = behaviour.str.extract(
                r"_(Small|Medium|Large)_value_payments$",
                expand=False,
            ).astype("object")
        else:
            data["Payment_Spend_Level"] = np.nan
            data["Payment_Value_Size"] = np.nan

        has_payment_data = delayed_payments.notna() | delay_from_due_date.notna()
        has_negative_history = delayed_payments.gt(0) | delay_from_due_date.gt(0)
        data["has_negative_payment_history"] = (
            has_negative_history.astype("float64").where(has_payment_data)
        )
        data["high_utilization_flag"] = (
            utilization.ge(35).astype("float64").where(utilization.notna())
        )
        data["low_balance_flag"] = (
            monthly_balance.le(monthly_salary.mul(0.1))
            .astype("float64")
            .where(monthly_balance.notna() & monthly_salary.notna())
        )
        data["debt_per_credit_product"] = self._safe_divide(
            outstanding_debt,
            total_credit_products,
        )

        for column in ENGINEERED_FEATURES:
            data[column] = (
                pd.to_numeric(data[column], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .astype("float64")
            )
        return data

    def _clip_numeric_features(self, data: pd.DataFrame) -> pd.DataFrame:
        for column, (lower, upper) in self.clip_bounds_.items():
            if column in data.columns:
                data[column] = pd.to_numeric(data[column], errors="coerce").clip(lower, upper)
        return data

    def _post_clean(self, data: pd.DataFrame) -> pd.DataFrame:
        data = data.drop(columns=[column for column in DROP_AFTER_CLEANING if column in data.columns])

        for column in data.columns:
            if pd.api.types.is_string_dtype(data[column]) or data[column].dtype == object:
                data[column] = data[column].astype("object").where(data[column].notna(), np.nan)

        return data

In [16]:
class CatBoostFeaturePreparer(BaseEstimator, TransformerMixin):
    # Готовит очищенные признаки к CatBoost без scaling и one-hot encoding.

    def __init__(self, categorical_fill_value: str = "Пропуск"):
        self.categorical_fill_value = categorical_fill_value

    def fit(self, X: pd.DataFrame, y: object = None):
        data = X.copy()
        self.output_columns_ = list(data.columns)
        self.numeric_feature_names_ = [
            column
            for column in self.output_columns_
            if pd.api.types.is_numeric_dtype(data[column])
        ]
        self.cat_feature_names_ = [
            column
            for column in self.output_columns_
            if column not in self.numeric_feature_names_
        ]
        self.cat_feature_indices_ = [
            self.output_columns_.index(column)
            for column in self.cat_feature_names_
        ]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        data = X.copy()

        for column in self.output_columns_:
            if column not in data.columns:
                data[column] = np.nan
        data = data[self.output_columns_]

        for column in self.numeric_feature_names_:
            data[column] = (
                pd.to_numeric(data[column], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .astype("float64")
            )

        for column in self.cat_feature_names_:
            data[column] = (
                data[column]
                .astype("object")
                .where(data[column].notna(), self.categorical_fill_value)
                .astype(str)
            )

        return data


preprocessing_pipeline = Pipeline(
    steps=[
        ("cleaner", CreditScoreCleaner(use_customer_profiles=True)),
        ("catboost_features", CatBoostFeaturePreparer()),
    ]
)


In [17]:
def split_features_target(data: pd.DataFrame, target: str = TARGET) -> tuple[pd.DataFrame, pd.Series]:
    X = data.drop(columns=[target])
    y = data[target].map(SCORE_MAP)

    if y.isna().any():
        unknown_values = sorted(data.loc[y.isna(), target].dropna().unique())
        raise ValueError(f"Неизвестные значения целевой переменной: {unknown_values}")

    return X, y.astype("int64")


def group_train_test_split(
    X: pd.DataFrame,
    y: pd.Series,
    group_column: str = GROUP_COLUMN,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    if group_column not in X.columns:
        raise ValueError(f"Колонка для группового split не найдена: {group_column}")

    groups = X[group_column]
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state,
    )
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    train_customers = set(X_train[group_column].dropna())
    test_customers = set(X_test[group_column].dropna())
    customer_overlap = train_customers.intersection(test_customers)
    if customer_overlap:
        raise ValueError(f"Leakage: клиентов одновременно в train и test: {len(customer_overlap)}")

    return X_train, X_test, y_train, y_test


X, y = split_features_target(train_raw)

X_train, X_test, y_train, y_test = group_train_test_split(X, y)

train_customers = set(X_train[GROUP_COLUMN].dropna())
test_customers = set(X_test[GROUP_COLUMN].dropna())
customer_overlap = train_customers.intersection(test_customers)

split_diagnostics = pd.DataFrame(
    {
        "metric": [
            "split_strategy",
            "train_rows",
            "test_rows",
            "train_customers",
            "test_customers",
            "customer_overlap",
        ],
        "value": [
            SPLIT_STRATEGY,
            len(X_train),
            len(X_test),
            len(train_customers),
            len(test_customers),
            len(customer_overlap),
        ],
    }
)
display(split_diagnostics)
display(y_train.value_counts(normalize=True).rename("train_share").to_frame().join(
    y_test.value_counts(normalize=True).rename("test_share")
))

X_train_prepared = preprocessing_pipeline.fit_transform(X_train, y_train)
X_test_prepared = preprocessing_pipeline.transform(X_test)

cleaner = preprocessing_pipeline.named_steps["cleaner"]
profile_customers = set(cleaner.customer_numeric_profiles_.index).union(
    set(cleaner.customer_categorical_profiles_.index)
)
profile_test_overlap = profile_customers.intersection(test_customers)
if profile_test_overlap:
    raise ValueError(f"Leakage: customer profiles содержат test-клиентов: {len(profile_test_overlap)}")

catboost_preparer = preprocessing_pipeline.named_steps["catboost_features"]
cat_feature_names = catboost_preparer.cat_feature_names_
cat_feature_indices = catboost_preparer.cat_feature_indices_
numeric_feature_names = catboost_preparer.numeric_feature_names_

engineered_feature_columns = [
    column
    for column in X_train_prepared.columns
    if column in ENGINEERED_FEATURES
    or any(column.startswith(f"{feature}_") for feature in ENGINEERED_CATEGORICAL_FEATURES)
]

numeric_missing_train = int(X_train_prepared[numeric_feature_names].isna().sum().sum())
numeric_missing_test = int(X_test_prepared[numeric_feature_names].isna().sum().sum())
categorical_missing_train = int(X_train_prepared[cat_feature_names].isna().sum().sum())
categorical_missing_test = int(X_test_prepared[cat_feature_names].isna().sum().sum())

missing_diagnostics = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "numeric_missing": [numeric_missing_train, numeric_missing_test],
        "categorical_missing": [categorical_missing_train, categorical_missing_test],
        "numeric_nan_policy": [NUMERIC_NAN_POLICY, NUMERIC_NAN_POLICY],
    }
)
display(missing_diagnostics)

print("X_train_prepared:", X_train_prepared.shape)
print("X_test_prepared:", X_test_prepared.shape)
print("Категориальных признаков:", len(cat_feature_names))
print("Engineered признаков:", len(engineered_feature_columns))
print("Пересечение Customer_ID train/test:", len(customer_overlap))
print("Пересечение customer profiles с test:", len(profile_test_overlap))
print("Числовые NaN оставлены намеренно для CatBoost:", numeric_missing_train, numeric_missing_test)
X_train_prepared.head()


,metric,value
0,split_strategy,customer_id_group_shuffle
1,train_rows,80000
2,test_rows,20000
3,train_customers,10000
4,test_customers,2500
5,customer_overlap,0


,train_share,test_share
Credit_Score,,
1,0.532425,0.5290
0,0.290900,0.2863
2,0.176675,0.1847


,dataset,numeric_missing,categorical_missing,numeric_nan_policy
0,train,32215,0,Оставляем числовые NaN для встроенной обработк...
1,test,71947,0,Оставляем числовые NaN для встроенной обработк...


X_train_prepared: (80000, 74)
X_test_prepared: (20000, 74)
Категориальных признаков: 7
Engineered признаков: 39
Пересечение Customer_ID train/test: 0
Пересечение customer profiles с test: 0
Числовые NaN оставлены намеренно для CatBoost: 32215 71947


,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,...,loan_diversity_ratio,credit_mix_ordinal,payment_of_min_amount_ordinal,payment_min_amount_yes_flag,Payment_Spend_Level,Payment_Value_Size,has_negative_payment_history,high_utilization_flag,low_balance_flag,debt_per_credit_product
0,January,23.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,3.0,...,1.0,2.0,2.0,0.0,High,Small,1.0,0.0,0.0,101.2475
1,February,23.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,NaN,...,1.0,2.0,2.0,0.0,Low,Large,1.0,0.0,0.0,101.2475
2,March,23.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,3.0,...,1.0,2.0,2.0,0.0,Low,Medium,1.0,0.0,0.0,101.2475
3,April,23.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,5.0,...,1.0,2.0,2.0,0.0,Low,Small,1.0,0.0,0.0,101.2475
4,May,23.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,6.0,...,1.0,2.0,2.0,0.0,High,Medium,1.0,0.0,0.0,101.2475


In [18]:
catboost_features_info = pd.DataFrame(
    {
        "cat_feature_name": cat_feature_names,
        "cat_feature_index": cat_feature_indices,
    }
)

display(catboost_features_info)
print("Всего признаков:", X_train_prepared.shape[1])
print("Первые engineered признаки:", engineered_feature_columns[:12])


,cat_feature_name,cat_feature_index
0,Month,0
1,Occupation,2
2,Credit_Mix,13
3,Payment_of_Min_Amount,16
4,Payment_Behaviour,19
5,Payment_Spend_Level,68
6,Payment_Value_Size,69


Всего признаков: 74
Первые engineered признаки: ['missing_marker_count', 'is_missing_Type_of_Loan', 'is_missing_Credit_History_Age', 'is_missing_Monthly_Inhand_Salary', 'is_missing_Credit_Mix', 'is_anomaly_Age', 'is_anomaly_Delay_from_due_date', 'anomaly_count', 'secured_loan_count', 'unsecured_loan_count', 'has_no_reported_loan_type', 'debt_to_annual_income']


In [19]:
prepared_diagnostics = pd.DataFrame(
    {
        "признак": X_train_prepared.columns,
        "роль": [
            "categorical" if column in cat_feature_names else "numeric"
            for column in X_train_prepared.columns
        ],
        "dtype": X_train_prepared.dtypes.astype(str),
        "пропуски": X_train_prepared.isna().sum().to_numpy(),
    }
)

prepared_diagnostics.sort_values(["роль", "пропуски", "признак"], ascending=[True, False, True]).head(40)


,признак,роль,dtype,пропуски
Credit_Mix,Credit_Mix,categorical,str,0
Month,Month,categorical,str,0
Occupation,Occupation,categorical,str,0
Payment_Behaviour,Payment_Behaviour,categorical,str,0
Payment_Spend_Level,Payment_Spend_Level,categorical,str,0
Payment_Value_Size,Payment_Value_Size,categorical,str,0
Payment_of_Min_Amount,Payment_of_Min_Amount,categorical,str,0
credit_age_per_loan,credit_age_per_loan,numeric,float64,9183
loan_diversity_ratio,loan_diversity_ratio,numeric,float64,9183
Age,Age,numeric,float64,4456


## Как переиспользовать с CatBoost

Замените `train_raw` на DataFrame с такой же raw-схемой, затем вызовите:

```python
X_new_prepared = preprocessing_pipeline.transform(X_new)
```

Для обучения CatBoost используйте `cat_feature_names` или `cat_feature_indices`.

```python
from catboost import CatBoostClassifier, Pool

train_pool = Pool(
    X_train_prepared,
    y_train,
    cat_features=cat_feature_names,
)

model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    random_seed=RANDOM_STATE,
)
model.fit(train_pool)
```

Важно:
- этот notebook не делает one-hot encoding и scaling;
- числовые `NaN` оставляются намеренно, потому что CatBoost умеет работать с пропусками в числовых признаках;
- категориальные пропуски заполняются строковой категорией;
- train/test split делается по `Customer_ID`, поэтому один клиент не должен попадать одновременно в train и test;
- если задача прогнозирует будущие месяцы, используйте time-based split по `Month` вместо текущего group split.


In [20]:
OUTPUT_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train_prepared.to_csv(
    OUTPUT_DIR / "X_train_catboost.csv",
    index=True,
    index_label="source_index",
)
X_test_prepared.to_csv(
    OUTPUT_DIR / "X_test_catboost.csv",
    index=True,
    index_label="source_index",
)
y_train.to_frame(name=TARGET).to_csv(
    OUTPUT_DIR / "y_train_catboost.csv",
    index=True,
    index_label="source_index",
)
y_test.to_frame(name=TARGET).to_csv(
    OUTPUT_DIR / "y_test_catboost.csv",
    index=True,
    index_label="source_index",
)

metadata = {
    "cat_feature_names": cat_feature_names,
    "cat_feature_indices": cat_feature_indices,
    "numeric_feature_names": numeric_feature_names,
    "target": TARGET,
    "score_map": SCORE_MAP,
    "split_strategy": SPLIT_STRATEGY,
    "group_column": GROUP_COLUMN,
    "test_size": TEST_SIZE,
    "train_rows": int(len(X_train_prepared)),
    "test_rows": int(len(X_test_prepared)),
    "train_customers": int(len(train_customers)),
    "test_customers": int(len(test_customers)),
    "customer_overlap": int(len(customer_overlap)),
    "customer_profile_test_overlap": int(len(profile_test_overlap)),
    "numeric_nan_allowed": True,
    "numeric_nan_policy": NUMERIC_NAN_POLICY,
    "numeric_missing_train": numeric_missing_train,
    "numeric_missing_test": numeric_missing_test,
    "categorical_missing_train": categorical_missing_train,
    "categorical_missing_test": categorical_missing_test,
    "time_based_split_note": "Если задача прогнозирует будущие месяцы, используйте split по Month: train на ранних месяцах, test на поздних.",
}
(OUTPUT_DIR / "catboost_feature_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Файлы сохранены в:", OUTPUT_DIR)
for file_name in [
    "X_train_catboost.csv",
    "X_test_catboost.csv",
    "y_train_catboost.csv",
    "y_test_catboost.csv",
    "catboost_feature_metadata.json",
]:
    print(OUTPUT_DIR / file_name)


Файлы сохранены в: C:\Users\stoli\PycharmProjects\Credit_score\data
C:\Users\stoli\PycharmProjects\Credit_score\data\X_train_catboost.csv
C:\Users\stoli\PycharmProjects\Credit_score\data\X_test_catboost.csv
C:\Users\stoli\PycharmProjects\Credit_score\data\y_train_catboost.csv
C:\Users\stoli\PycharmProjects\Credit_score\data\y_test_catboost.csv
C:\Users\stoli\PycharmProjects\Credit_score\data\catboost_feature_metadata.json
